In [4]:
# Weather Dashboard Version 1.3.5
# This version is the completedly backbone of the fetching application,
# it displays basic weather variables and is simply an input-output model. 

In [ ]:
# first im importing all nessary libraries, to highlight, QPushButton,QLineEdit, and QWidget were the most vital parts of this introductory code.


In [6]:
import datetime as datetime_
import sys
import requests
from PyQt5.QtWidgets import (QApplication, QLabel, QVBoxLayout, QWidget,
                             QLineEdit, QPushButton, QFrame, QListWidget, QListWidgetItem)
from PyQt5.QtCore import Qt, QTimer

In [ ]:
# first below i am setting the defining the QWidget subclass for WeatherDashboard,
# which gives the application "top-level" a QWidget, giving WeatherDashboard executing proirity over the main proceeding script. 
# This allows me to extend the code/behavoir with my custom functionalities, Hence why its first.

# Why QWidget? Primarily its capabilities of data display, received user input, provides a container for widgets, and status information. 
# Since my widgets deal with graphical elements, QWidget provides the necessary infrastructure to manage and display them effectively.

In [ ]:
class WeatherDashboard(QWidget):
    def __init__(self):     #   Here I arrange a definition to "initialize" since __init__ is the constructor method for the class, it'll run the instance of WeatherDashboard
        super().__init__()      # Call the constructor of the parent class (QWidget) to initialize the widget properly, in this case super returns a parent proxy for the WeatherDashboard instance.
                                    # calling it in this way is useful if class hierarchy changes or if the parent class implementation needs to be preserved.
        self.clock_label = QLabel(self)     #QLabel creates the text/ display widget, I assign that widget to the instance variable self.clock_label so other methods in the class can access and update it as needed.
        self.city_label = QLabel("Enter City Name: ", self)  # Label prompting the user to enter the city name & the corresponding input field is self.city_input
        self.city_input = QLineEdit(self)
        self.get_weather_button = QPushButton("Get Weather", self) # QPushButton creates a clickable button widget, this behaves on the Qt signal-slot mechanism, allowing you to connect it to a function that will be executed when the button is clicked.
        self.temperature_label = QLabel("Temperature: ", self)  # Another label for Temperature 
        self.emoji_label = QLabel("", self)                 # An emoji label with a "" string initially, will be updated with weather emoji later with the API fetches the Weather Code
        self.description_label = QLabel("Description: ", self) 
        self.wind_speed_label = QLabel("Wind Speed: ", self)
        self.init_ui()          # Now im calling my own method, meaning init_ui will handle the visual configuration and layout of the interface seperate from the initialization.
        
    def init_ui(self):  # A seperate init_ui method will now configure the presentation layer of the dashboard and visual behavoir.
        self.setAttribute(Qt.WA_TranslucentBackground)  # [error] never displayed. 09/26/26 TBH
        self.setWindowTitle("Weather Dashboard")    #setting display title of the window
        self.setMinimumWidth(420)       # Minimun allowable width of window 
        layout = QVBoxLayout()          # this function is a vertical box manager, it'll arrange child widgets in a single-column layout
        layout.addWidget(self.clock_label)      # adding the clock label to the layout as its own widget x7
        layout.addWidget(self.city_label)
        layout.addWidget(self.city_input)
        layout.addWidget(self.get_weather_button, alignment=Qt.AlignCenter)    # Qt.AlignCenter aligns the button to the center of the layout, So here im actually overriding the default alignment behavior of the layout for this specific widget.
        layout.addWidget(self.temperature_label)
        layout.addWidget(self.emoji_label)
        layout.addWidget(self.description_label)
        layout.addWidget(self.wind_speed_label)
        
        self.setLayout(layout) # set the layout for the main widget

In [ ]:
  # Hide weather results until a valid city is searched
   #adding parameter setVisible(False) before valid city is search; essentially hiding actualized widgets
self.temperature_label.setVisible(False)    # typer: ignore # 
self.wind_speed_label.setVisible(False)    # type: ignore #
self.description_label.setVisible(False)    # type: ignore #


In [ ]:
# Here I use Qt.AlignCenter again to display all of my labels and input fields in the center vertically descending in script order
        self.city_label.setAlignment(Qt.AlignCenter)
        self.clock_label.setAlignment(Qt.AlignCenter)
        self.city_input.setAlignment(Qt.AlignCenter)
        self.temperature_label.setAlignment(Qt.AlignCenter)
        self.emoji_label.setAlignment(Qt.AlignCenter)
        self.description_label.setAlignment(Qt.AlignCenter)
        self.wind_speed_label.setAlignment(Qt.AlignCenter)
        self.get_weather_button.setAlignment(Qt.AlignCenter)
        # Now set the object names for all the widgets, which is useful for styling with Qt Style Sheets (QSS) and for debugging purposes
         self.city_label.setObjectName("city_label")
        self.clock_label.setObjectName("clock_label")
        self.city_input.setObjectName("city_input")
        self.get_weather_button.setObjectName("get_weather_button")
        self.temperature_label.setObjectName("temperature_label")
        self.emoji_label.setObjectName("emoji_label")
        self.description_label.setObjectName("description_label")
        self.wind_speed_label.setObjectName("wind_speed_label")

In [ ]:
# Stylesheet for the weather dashboard application
        self.setStyleSheet("""
            QLabel, QPushButton, QLineEdit {
                font-family: Arial;
            }
            QWidget {
            background: transparent;
            }

            QFrame {
           background-color: rgba(255, 255, 255, 0);
            border-radius: 25px;
            }
            QLabel#city_label {
                font-size: 40px;
            }
            QLabel#clock_label {
                font-size: 16px;
                color: #555555;
            }
            QLineEdit {
                padding: 12px;
                border: 3px solid #aaaaaa;
                border-radius: 4px;
            }
            QPushButton {
                padding: 12px 24px;
                color: white;
                background-color: #e550eb;
                border: none;
                border-radius: 3px;
            }
            QPushButton:hover {
                background-color: #6250eb;
            }
            QLabel#temperature_label, QLabel#description_label {
                font-size: 25px;
            }
            QLabel#emoji_label {
                font-size: 100px;
                font-family: 'Apple Color Emoji';
            }
            QLabel#wind_speed_label {
                font-size: 15px;
            }
        """)

In [ ]:
     self.get_weather_button.clicked.connect(self.get_weather) #type: ignore # Setting the class for the button click, allowing the weather retrieval event to launch, finding the slot method to handle the click event 
     self.clock_timer = QTimer(self)    # QTimer for updating the clock every second
     self.clock_timer.timeout.connect(self.update_clock) # Connect the timer's timeout signal to the update_clock method
     self.clock_timer.start(1000)            # Start the timer with a 1-second interval, necessary for real-time clock updates, i.e updating the clock every second
     self.update_clock() # Initial clock update 

In [ ]:
 def update_clock(self):
       current_time = datetime_.datetime.now() # Get the current date and time, using the datetime.now() to update the clock display
       self.clock_label.setText(current_time.strftime("%A, %B %d | %I:%M:%S %p")) # Update the clock label with the current date and time using this specific format: Day of the week, Month, Day | Hour:Minute:Second AM/PM

In [ ]:
def get_weather(self):  # defining the method to fetch weather data for the specified city
        api_key = "acbce1234567898765000hijklmnop"
        city = self.city_input.text().strip()   # Get the city name entered by the user and recieve as a string

In [ ]:
# setting up my first if/else/execpt function for retrieval errors and displaying appropriate messages to the user if city not found 
  if not city:      # Check if the city input is empty
            self.display_error("Please enter a city name.")
            return

        url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"   # Construct API URL into an f string with the city and API key, f strings are good for readability and avoiding string concatenation
        try:    # Attempt to retrieve weather data from the API
            response = requests.get(url, timeout=10)  # Send GET request to the API with a 10-second timeout, timeout is good for making sure the request doesn't hang indefinitely
            response.raise_for_status() # raise_for_status will raise an HTTPError if the HTTP request returned an unsuccessful status code
            data = response.json()      # Parse the JSON response into a Python dictionary
        except (requests.RequestException, ValueError) as error: # Handle network errors and JSON parsing errors
            self.display_error(f"Unable to retrieve weather: {error}") # 
            return

        if response.ok and data.get("cod") == 200: # Check if the response is successful and the city was found, specifically ("cod") == 200 meaning Valid response from the API
            # now i convert the values to metric for kelvin and m/s
            temperature = (data["main"]["temp"] - 273.15) * 9 / 5 + 32 # getting the main temp out of all the temp variables
            wind_speed = data["wind"]["speed"] * 2.237      # converting the wind speed from m/s to mph out of the wind variables
            description = data["weather"][0]["description"].capitalize()    # "[0]" - Access the first element of the weather list and capitalize the description for better readability
            weather_id = data["weather"][0]["id"]   # Get the weather condition ID for determining the appropriate emoji
            
            self.display_weather(   # Display the retrieved weather information to the user if the API call was successful and the city was found
                temperature,
                description,
                wind_speed,
                self.get_weather_emoji(weather_id),
            )
        else:       # else - the API call was not successful or the city was not found, display an error message
            self.display_error(data.get("message", "City not found."))


In [ ]:
    def get_weather_emoji(self, weather_id): # Returns an emoji representing the weather condition based on the weather ID from the OpenWeatherMap API
        if weather_id in range(200, 233): # thunderstorm 200-232
            return "⛈️"
        if weather_id in range(300, 322):   # light rain/drizzle 300-321
            return "🌦️"
        if weather_id in range(500, 532):   # rain 500-531, room for error as some IDs might not be covered exactly
            return "🌧️"
        if weather_id in range(600, 623):   # snow 600-622, room for error as some ID's arent covered just like rain
            return "❄️"
        if weather_id in range(700, 782):   # atmosphere 700-781, includes mist, smoke, haze, etc.
            return "🌫️"
        if weather_id == 800:                #sunny/ clear conditions 
            return "☀️"
        if weather_id in range(801, 805):   # clouds 801-805, covered by partly cloudy emoji
            return "⛅"
        return "🌤️" 


In [ ]:
def display_error(self, message):       # definition to Display an error message on the weather dashboard from the previous self.display_error exception script
        self.temperature_label.clear() # clear the variables from displaying previous weather data
        self.wind_speed_label.clear()
        self.temperature_label.setVisible(False)        #clear visibility 
        self.wind_speed_label.setVisible(False)
        self.description_label.setVisible(True) #keep description and 

        self.emoji_label.setText("⚠️")
        self.description_label.setText(f"Error: {message}") # Display the error message on the description label

In [ ]:
 def display_weather(self, temperature, description, wind_speed, emoji): # Display the weather information on the dashboard
        self.temperature_label.setVisible(True)
        self.wind_speed_label.setVisible(True)
        self.description_label.setVisible(True)

        self.temperature_label.setText(f"Temperature: {temperature:.1f}°F")
        self.emoji_label.setText(emoji)
        self.description_label.setText(f"Description: {description}")
        self.wind_speed_label.setText(f"Wind Speed: {wind_speed:.1f} mph")

In [ ]:
if __name__ == "__main__":
    app = QApplication(sys.argv)
    dashboard = WeatherDashboard()
    dashboard.show()
    sys.exit(app.exec_())